<a href="https://colab.research.google.com/github/nsasto/echo/blob/main/goEcho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Upload our kaggle.json API file


In [1]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nathansasto","key":"fc6550be85a309f54202b321a3ac2ee7"}'}

In [2]:
# Create a directory for Kaggle configuration if it doesn't exist
!mkdir -p ~/.kaggle/

# Copy the uploaded kaggle.json file to the Kaggle configuration directory
!cp kaggle.json ~/.kaggle/

# Set appropriate permissions for the kaggle.json file
# This is crucial for security and Kaggle API functionality (read/write by owner only)
!chmod 600 ~/.kaggle/kaggle.json

# Install the Kaggle API client (if not already installed)
!pip install -q kaggle

Download the dataset (our custom whisper model)

In [3]:

# Download the dataset. The -d flag specifies a dataset.
# The dataset will be downloaded as a zip file to the current working directory (/content/ by default).
!kaggle datasets download -d nathansasto/whisper-echo

Dataset URL: https://www.kaggle.com/datasets/nathansasto/whisper-echo
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
 94% 805M/854M [00:04<00:01, 46.0MB/s]
100% 854M/854M [00:04<00:00, 199MB/s] 


In [4]:
# Create a directory for the unzipped contents
!mkdir -p whisper_echo

# Unzip the downloaded file into the new directory
!unzip whisper-echo.zip -d whisper_echo

Archive:  whisper-echo.zip
  inflating: whisper_echo/added_tokens.json  
  inflating: whisper_echo/config.json  
  inflating: whisper_echo/generation_config.json  
  inflating: whisper_echo/merges.txt  
  inflating: whisper_echo/model.safetensors  
  inflating: whisper_echo/normalizer.json  
  inflating: whisper_echo/preprocessor_config.json  
  inflating: whisper_echo/special_tokens_map.json  
  inflating: whisper_echo/tokenizer_config.json  
  inflating: whisper_echo/training_args.bin  
  inflating: whisper_echo/vocab.json  


In [5]:
import os
import torch
import pandas as pd
from datasets import Dataset, Audio
from transformers import (
    pipeline,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch

Install dependency requirements for pyaudio in colab

In [7]:
%%capture
#in colab
!apt-get install -y portaudio19-dev
!pip install pyaudio